In [2]:
version = "REPLACE_PACKAGE_VERSION"

# Experiment Design and Analysis
## School of Information, University of Michigan

## Week 4: 
- 1. Threats to Validity
- 2. Instrumental Variables

## Assignment Overview
### The objective of this assignment is to:

- Apply theory of experiment design and knowledge of analysis techniques to real experiment data.


### The total score of this assignment will be 25 points


### Resources:
- StatsModels
    - We recommend using a python library called [StatsModels](https://www.statsmodels.org/stable/index.html) for data analysis


- Dataset used in this assignment: Kiva Crowdsourcing Team data [view source files](https://www.openicpsr.org/openicpsr/project/100358/version/V2/view)
    - Source for dataset: [Chen, Y., et al. Recommending teams promotes prosocial lending in online microfinance (2016).](https://www.pnas.org/content/113/52/14944)

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS #you may get a deprecation warning for this library -- this is fine.
from mads.lib.path import assets

file = assets.find("assignment4_data.csv")
data = pd.read_csv(file) #Data for this assignment

In [6]:
#uncomment the below line to view readme files for this dataset (includes explanation of variable names)
file_readme = assets.find("assignment4_data_readme.md")
!cat $file_readme

#uncomment the below line to view snippet of csv file
data.head()

### Assignment Topic: Data analysis of a field experiment on Kiva

### Background:

We upload data files from a field experiment conducted by the University of Michigan on the Kiva platform. They are in csv format to import into the Jupyter Notebook.

### Data:

The data file contains eight variables for 64,800 subjects. Below are descriptions of each field in the file:

    - shuffled_lender_id: a de-identified identifier to represent unique subjects.
    
    - treatment_id: the type of treatment the subjects received as part of the experiment.
    
        1. No-Contact
        
        2. Team-Exist
        
        3. Location-Explanation
        
        4. Location-NoExplanation
        
        5. History-Explanation
        
        6. History-NoExplanation
        
        7. Leaderboard-Explanation
        
        8. Leaderboard-NoExplanation
        
    - join: Whether a lender joined a team after the experiment.
    
    - join_rec: Whether a lender joined a recommended 

,shuffled_lender_id,treatment_id,join,join_rec,opened,amt_diff_1d,amt_diff_7d,amt_diff_30d
0,0,7,0,0,1,0.0,0.0,-50.0
1,1,3,0,0,1,0.0,0.0,0.0
2,2,4,0,0,1,0.0,0.0,0.0
3,3,2,0,0,0,0.0,0.0,0.0
4,4,5,0,0,0,0.0,0.0,0.0


## Part A (15 points)

We want to assess the effectiveness of joining a team on Kiva -- specifically, what impact joining a team has on donations. Using the variable indicating whether users have joined a team (```join```) and the differences in donations made over a certain period (```amt_diff_1d```, ```amt_diff_7d```, ```amt_diff_30d```), we can find if joining a team has impact on donations. However, variables that determine whether subjects join a team may also affect the amount they donate since we only inform subjects about joining a team.

In this case, our instrumental variable is whether the subjects were sent an e-mail to inform about the team functionality on Kiva. 

***Before you go on, recall from lecture the requirements an instrumental variable must satisfy - you will be investigating these requirements in this notebook.***

1. To get started, we need to create the instrumental variable. Add a new column named ```email``` in the dataframe. The value for ```email``` should be ```1``` if users received an e-mail as part of their treatment group (```treatment_id``` != 1), and ```0``` if they did not (```treatment_id``` = 1). (2 points)

In [ ]:
# Rename the original outcome so it does not conflict with pandas.DataFrame.join().
# This keeps the notebook code readable and avoids accidental method/column confusion later on.
data.rename(columns={'join': 'join_any'}, inplace=True)

# Construct the instrument used throughout the assignment.
# Treatment group 1 received no email, so it is coded as 0.
# All other treatment groups received an email, so they are coded as 1.
# This variable captures random encouragement rather than actual team joining behavior.
data['email'] = np.where(data['treatment_id'] == 1, 0, 1)


In [ ]:
assert pd.notnull(data['email'].all()), "email column must contain either 0 or 1"

In [ ]:
assert data.loc[data['treatment_id'] != 1,'email'].all() == 1, "all treatments except treatment 1 received an email"
assert data.loc[data['treatment_id'] == 1,'email'].all() == 0, "all treatments except treatment 1 received an email"

2. Next, we will create a constant, equal to 1. Add a column in the dataframe called ```const```. (1 point)

In [ ]:
# Add an intercept column explicitly because the models in this assignment
# are being fit from design matrices we build by hand.
# Statsmodels OLS does not add a constant automatically when we pass X directly.
data['const'] = 1


In [ ]:
assert data['const'].all() == 1, "the constant value should be 1"

In lecture, the 2-stage least squares model was used. Now, let’s follow the steps indicated in lecture to create this model to measure the effect described above. First, we need to estimate the effect of e-mailing users to join a team on whether they join a team.


### Why this workflow works

This assignment uses an **instrumental variables / two-stage least squares (2SLS)** design.

- **Endogenous variable:** `join_any`  
  People choose whether to join a team, so joining may be correlated with unobserved motivation.

- **Instrument:** `email`  
  The encouragement email changes the likelihood of joining, but is intended to affect lending only through joining.

- **Stage 1:** predict joining from the instrument  
- **Stage 2:** use that predicted joining behavior to estimate the effect on lending outcomes

The code comments below explain the rationale for each implementation step so the notebook is easier to study later.


3. Using statsmodels, create an ordinary least squares regression model that does this. Fit the model and store it in the variable: ```model_fs```. Using the predict method from ```model_fs```, store the predicted values in a new column in your dataframe called ```predicted_join```. Recall from lecture that since we have created this new variable, we can estimate the effect of joining a team on lending amounts without worrying about the effect of potential unobserved or missing variables. (4 points)

Note: ensure your model has a constant.

In [ ]:
def email_join_ols(provided_data):
    global model_fs

    # First stage of 2SLS:
    # Regress actual team joining (endogenous variable) on the instrument (email encouragement).
    # The goal is to isolate only the variation in joining that is explained by the randomized email.
    X = provided_data[['const', 'email']]
    y = provided_data['join_any']

    # Fit the first-stage model and keep it in a global variable because the notebook's
    # later checks may expect the fitted object to be accessible by this exact name.
    model_fs = sm.OLS(y, X).fit()

    # Store the fitted values from the first stage.
    # 'predicted_join' is the part of joining behavior explained by the instrument,
    # and it is what we carry into the second stage instead of the raw join_any column.
    provided_data['predicted_join'] = model_fs.predict(X)

    return provided_data


Your function should return a dataframe with the correct values and columns. Check that it does:

In [ ]:
data = email_join_ols(data)
data.head()  # you may get a deprecation warning here -- that is ok.


In [ ]:
assert 'predicted_join' in data, "checking there is a column named predicted_join in data"

In [ ]:
"""checking the correct predicted_join values are present"""
# Hidden tests

Now that we have the predicted values of whether a subject would be expected to join a team based on if they were e-mailed, we can move to the second stage.

4. In this stage, we will run the estimation of the effect of joining a team on the amount a subject lends. However, instead of using the ```join``` variable, we will use our new ```predicted_join``` variable. Using statsmodels again, and ensuring your model has a constant, create three ordinary least squares regression models which estimate the effect of the prediction of users joining a team on the following:

a. ```amt_diff_1d```, storing the fitted model in ```model_1d``` (3 points)

In [ ]:
def pred_join_amt_1d(provided_data):
    # Work on a copy so the function does not accidentally mutate the caller's dataframe.
    # This is a good defensive habit in notebooks where cells may be run out of order.
    provided_data = provided_data.copy()

    # Second stage of 2SLS:
    # Regress 1-day lending change on the instrument-driven component of joining.
    # This estimates the causal effect of joining using only the exogenous variation
    # induced by the email encouragement.
    X = provided_data[['const', 'predicted_join']]
    y = provided_data['amt_diff_1d']

    model_1d = sm.OLS(y, X).fit()
    return model_1d


Your function should return a summary view of your results. Check that it does:

In [ ]:
print(pred_join_amt_1d(data).summary()) #we've wrapped this in a print statement to preserve the original statsmodels layout. You may get a deprecation warning here -- that is ok.

In [ ]:
"""checking your t-value is correct"""
# Hidden tests

b. ```amt_diff_7d```, storing the fitted model in ```model_7d``` (3 points)

In [ ]:
def pred_join_amt_7d(provided_data):
    # Use a copy to avoid modifying the original dataframe inside the function.
    provided_data = provided_data.copy()

    # Same second-stage setup as above, but with the 7-day lending outcome.
    X = provided_data[['const', 'predicted_join']]
    y = provided_data['amt_diff_7d']

    model_7d = sm.OLS(y, X).fit()
    return model_7d


Your function should return a summary view of your results. Check that it does:

In [ ]:
print(pred_join_amt_7d(data).summary())

In [ ]:
"""checking your t-value is correct"""
# Hidden tests

c. ```amt_diff_30d```, storing the fitted model in ```model_30d``` (2 points)

In [ ]:
def pred_join_amt_30d(provided_data):
    # Use a copy to keep the function side-effect free.
    provided_data = provided_data.copy()

    # Final second-stage model for the 30-day lending outcome.
    # Comparing 1d, 7d, and 30d lets us see whether the estimated effect persists over time.
    X = provided_data[['const', 'predicted_join']]
    y = provided_data['amt_diff_30d']

    model_30d = sm.OLS(y, X).fit()
    return model_30d


Your function should return a summary view of your results. Check that it does:

In [ ]:
print(pred_join_amt_30d(data).summary())

In [ ]:
"""checking your t-value is correct"""
# Hidden tests

### Part B rationale

Part B estimates the same causal relationship with `IV2SLS` directly instead of manually performing separate first- and second-stage OLS models.  
This is useful because it mirrors the econometrics notation more closely and reduces the chance of implementation mistakes.


## Part B (10 points)

Now we have estimated the effect of joining a team on lending using instrumental variables! However, there is a more direct way to complete these two stages.

Using the IV2SLS (Instrumental Variables 2-Stage Least Squares) function ([Documentation](https://bashtage.github.io/linearmodels/iv/iv/linearmodels.iv.model.IV2SLS.html)) in the linearmodels library will achieve everything we did above faster -- and it will more correctly estimate the standard errors.

1. First, let's make things simpler for ourselves. For this analysis, we will only need a dataframe with the following columns: ```email```, ```join_any```, ```amt_diff_1d```, ```amt_diff_7d```, ```amt_diff_30d```, and the ```const``` column you created above. (As in our other models, we need a constant included in the models we will be creating with IV2SLS.) (1 point)

In [ ]:
iv_dataframe = pd.DataFrame()

# Build a clean analysis dataframe containing only the variables required by IV2SLS.
# Keeping just the relevant columns makes the model specification easier to read and debug:
# - email: instrument
# - join_any: endogenous treatment variable
# - amt_diff_*: outcomes
# - const: exogenous intercept
iv_dataframe = data[['email', 'join_any', 'amt_diff_1d', 'amt_diff_7d', 'amt_diff_30d', 'const']].copy()


```iv_dataframe``` should yield a dataframe with the correct calculated, given, and new column and row values. Check that it does:

In [ ]:
iv_dataframe.head()

In [ ]:
"""checking your dataframe columns are all present"""
assert 'email' and 'join_any'and 'amt_diff_1d' and 'amt_diff_7d' and 'amt_diff_30d'and 'const' in iv_dataframe

After looking over the documentation of the IV2SLS function, create and fit three models (with the three lending measurement periods used in number 3) that estimate the effect of joining a team on lending amounts considering the instrument of emailing subjects about joining a team.

According to the [linearmodels IV2SLS documentation](https://bashtage.github.io/linearmodels/iv/iv/linearmodels.iv.model.IV2SLS.html), you will need to provide a given set of parameters to the function in order to create the model. The dependent variables and instruments are straightforward, but what are exogenous and endogenous regressors?

An exogenous regressor does not co-vary with the model’s random error while an endogenous regressor does. In our model’s case, we know the ```join_any``` variable co-varies with the random error while ```const``` cannot (since it is a constant!).

2. Create and fit the model for the first lending measurement period, the period referred to in ```amt_diff_1d``` (3 points)

In [ ]:
def iv_model_1d(provided_data):

    """Fit the direct IV2SLS model for the 1-day lending outcome."""
    # Copy the dataframe so the function remains safe to rerun.
    provided_data = provided_data.copy()

    # Direct IV formulation:
    # dependent   = lending outcome
    # exog        = intercept only
    # endog       = actual team joining (potentially endogenous)
    # instruments = email encouragement
    #
    # Using IV2SLS here should recover the same basic causal logic as the manual
    # two-stage approach from Part A, but in a more compact and standard form.
    iv_result_1d = IV2SLS(
        dependent=provided_data['amt_diff_1d'],
        exog=provided_data[['const']],
        endog=provided_data['join_any'],
        instruments=provided_data['email']
    ).fit(cov_type='unadjusted')

    return iv_result_1d


Your function should return a summary view of your results. Check that it does:

In [ ]:
iv_model_1d(iv_dataframe)

In [ ]:
"""checking your 1d model has an unadjusted covariance and your p-value is correct"""
# Hidden tests

3. Create and fit the model for the first lending measurement period, the period referred to in ```amt_diff_7d``` (3 points)

In [ ]:
def iv_model_7d(provided_data):
    # Copy the dataframe so repeated calls do not alter shared state.
    provided_data = provided_data.copy()

    # Same IV structure as the 1-day model, now using the 7-day outcome.
    iv_result_7d = IV2SLS(
        dependent=provided_data['amt_diff_7d'],
        exog=provided_data[['const']],
        endog=provided_data['join_any'],
        instruments=provided_data['email']
    ).fit(cov_type='unadjusted')

    return iv_result_7d


Your function should return a summary view of your results. Check that it does:

In [ ]:
iv_model_7d(iv_dataframe)

In [ ]:
"""checking your 7d model has an unadjusted covariance and your p-value is correct"""
# Hidden tests

4. Create and fit the model for the first lending measurement period, the period referred to in ```amt_diff_30d``` (3 points)

In [ ]:
def iv_model_30d(provided_data):
    # Copy the dataframe so this function remains reproducible and side-effect free.
    provided_data = provided_data.copy()

    # Same IV structure for the 30-day outcome.
    iv_result_30d = IV2SLS(
        dependent=provided_data['amt_diff_30d'],
        exog=provided_data[['const']],
        endog=provided_data['join_any'],
        instruments=provided_data['email']
    ).fit(cov_type='unadjusted')

    return iv_result_30d


Your function should return a summary view of your results. Check that it does:

In [ ]:
iv_model_30d(iv_dataframe)

In [ ]:
"""checking your 30d model has an unadjusted covariance and your p-value is correct"""
# Hidden tests